In [3]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("./data/sql-murder-mystery.db")

def q(query):
    """Función helper: ejecuta una query y devuelve un DataFrame."""
    return pd.read_sql(query, conn)

# Ver las tablas disponibles
q("SELECT name FROM sqlite_master WHERE type='table'")

,name
0,crime_scene_report
1,drivers_license
2,person
3,facebook_event_checkin
4,interview
5,get_fit_now_member
6,get_fit_now_check_in
7,income
8,solution


## Buscamos la información del crimen

In [4]:
q("""
    SELECT *
    FROM crime_scene_report
    WHERE date = 20180115
      AND type = 'murder'
      AND city = 'SQL City'
""")

,date,type,description,city
0,20180115,murder,Security footage shows that there were 2 witne...,SQL City


Description : Security footage shows that there were 2 witnesses. The first witness lives at the last house on "Northwestern Dr". The second witness, named Annabel, lives somewhere on "Franklin Ave".

Hay dos testigos/witness

## Los identificamos

In [5]:
q("""
    SELECT *
    FROM person 
    WHERE address_street_name = "Northwestern Dr"
    ORDER BY address_number DESC
    LIMIT 1
""")

,id,name,license_id,address_number,address_street_name,ssn
0,14887,Morty Schapiro,118009,4919,Northwestern Dr,111564949


"14887/ Morty Schapiro"

In [6]:
q("""
    SELECT *
    FROM person
    WHERE name LIKE "Annabel%"
        AND address_street_name = "Franklin Ave"
""")

,id,name,license_id,address_number,address_street_name,ssn
0,16371,Annabel Miller,490173,103,Franklin Ave,318771143


"16371/ Annabel Miller"

## Vemos su entrevista

In [10]:
q("""
    SELECT *
    FROM interview
    WHERE person_id = 14887 
        OR person_id = 16371
""")

,person_id,transcript
0,14887,I heard a gunshot and then saw a man run out. ...
1,16371,"I saw the murder happen, and I recognized the ..."


-Morty: "I heard a gunshot and then saw a man run out. He had a "Get Fit Now Gym" bag. The membership number on the bag started with "48Z". Only gold members have those bags. The man got into a car with a plate that included "H42W"."

-Annabel: "I saw the murder happen, and I recognized the killer from my gym when I was working out last week on January the 9th."

Membresia = 487%, gold member, Matricula = %H42W%
date = January the 9th

 ## Buscamos coincidencias

In [15]:
q("""
    SELECT p.id, p.name, g.id AS gym_membership_id, g.membership_status
    FROM get_fit_now_member g
    JOIN person p ON g.person_id = p.id
    WHERE g.id LIKE "48Z%"
        AND g.membership_status = 'gold'
""")

,id,name,gym_membership_id,membership_status
0,28819,Joe Germuska,48Z7A,gold
1,67318,Jeremy Bowers,48Z55,gold


Dos nombre: Joe Germuska, Jeremy Bowers

Hojeamos las matriculas de coche

In [20]:
q("""
    SELECT p.id, p.name, d.plate_number
    FROM person p
    JOIN drivers_license d ON p.license_id = d.id
    WHERE d.plate_number LIKE "%H42W%"
""")

,id,name,plate_number
0,51739,Tushar Chandra,4H42WR
1,67318,Jeremy Bowers,0H42W2
2,78193,Maxine Whitely,H42W0X


Reaparece Jeremy Bowers ( Ser tocayo de Joe Goldberg no te hace asesino)

Confirmamos su presencia con la fecha que dieron

In [25]:
q("""
    SELECT p.name, c.check_in_date, c.check_in_time, c.check_out_time, g.id, g.membership_status
    FROM get_fit_now_check_in c
    JOIN get_fit_now_member g ON c.membership_id = g.id
    JOIN person p ON p.id = g.person_id
    WHERE c.check_in_date = 20180109
        AND g.membership_status = 'gold'
""")

,name,check_in_date,check_in_time,check_out_time,id,membership_status
0,Sarita Bartosh,20180109,486,1124,XTE42,gold
1,Burton Grippe,20180109,399,515,6LSTG,gold
2,Carmen Dimick,20180109,367,959,GE5Q8,gold
3,Joe Germuska,20180109,1600,1730,48Z7A,gold
4,Jeremy Bowers,20180109,1530,1700,48Z55,gold
5,Annabel Miller,20180109,1600,1700,90081,gold


Vemos que Jeremy Bowers estubo cuando pasó, apuntandole como principal sospechoso. De todas formas si Jeremy no hubiera aparecido Joe Germuska también estubo presente y pudo haber habido una confución en el testimonio de Morty Schapiro

## Vemos la interviuw de Jeremy
(La web nos confirma que Jeremy es un asesino, pero nos dice que todavía quedan cosas por hacer... Por lo visto Jeremy era un sicario)

In [28]:
q("""
    SELECT p.name, i.transcript
    FROM interview i
    JOIN person p ON i.person_id = p.id
    WHERE p.name = "Jeremy Bowers"
  """)

,name,transcript
0,Jeremy Bowers,I was hired by a woman with a lot of money. I ...


Jeremy: I was hired by a woman with a lot of money. I don't know her name but I know she's around 5'5" (65") or 5'7" (67"). She has red hair and she drives a Tesla Model S. I know that she attended the SQL Symphony Concert 3 times in December 2017.

Mide entre 65"-67", Charo, Coche = Tesla Model S, 3 veces en el SQL Symphony Concert en Diciembre 2017 (Sabe dónde estubo en el mes pasado y el coche, pero no la matricula...)

In [35]:
q("""
    SELECT p.name, d.gender, d.height, d.hair_color, d.car_model, inc.annual_income
    FROM person p
    JOIN drivers_license d ON p.license_id = d.id
    JOIN income inc ON p.ssn = inc.ssn
    WHERE d.gender = "female"
      AND (d.height > 65 AND d.height < 67)
      AND d.hair_color = "red"
      AND d.car_make = "Tesla"
      AND d.car_model = "Model S"
      AND p.id IN (
          SELECT person_id
          From facebook_event_checkin
          Where event_name = "SQL Symphony Concert"
            AND date LIKE "201712%"
          GROUP BY person_id
          HAVING COUNT(*) > 1
      )
""")

,name,gender,height,hair_color,car_model,annual_income
0,Miranda Priestly,female,66,red,Model S,310000


Miranda Priestly, la villana del Diablo viste de Prada es la mente criminal de todo este caso, lo típico.

In [ ]:
q("""
    SELECT p.name, i.transcript
    FROM interview i
    JOIN person p ON i.person_id = p.id
    WHERE p.name = "Miranda Priestly"
""")
## No le interogaron

,name,transcript
